# ko-pii 33종 PII 검출 테스트 샘플

- 대상 라이브러리: [Marker-Inc-Korea/ko-pii](https://github.com/Marker-Inc-Korea/ko-pii)
- 목적: **장문 문서**를 입력했을 때 주민번호 / 연락처 / 이메일 / 약품번호 등 **33종 PII**가 검출되는지 확인하고,
  검출된 각 항목의 **원문 좌표(start:end)** 를 함께 출력한다.

> ko-pii 는 외부 ML 의존성이 없고(표준 라이브러리만 사용) 규칙/사전/체크섬 기반으로 동작해
> 폐쇄망/오프라인에서도 쓸 수 있는 한국어 PII 탐지·비식별화 라이브러리다.

각 `Detection` 객체는 다음 좌표/속성을 제공한다.

| 속성 | 의미 |
|------|------|
| `label` | PII 종류 (예: `RRN`, `PHONE`) |
| `text` | 원문에서 매칭된 실제 문자열 |
| `start` | 원문 기준 시작 문자 오프셋(좌표) |
| `end` | 원문 기준 끝 문자 오프셋(좌표) |
| `confidence` | 신뢰도 0.0 ~ 1.0 |
| `evidence` | 매칭 근거(법적 근거/규칙) |

## 1) 설치

이미 설치되어 있으면 아래 셀은 건너뛰어도 된다. (파일 파싱 확장은 `ko-pii[file]`)

In [1]:
# ko-pii 설치 (이미 설치돼 있으면 재실행해도 무해)
import sys
!{sys.executable} -m pip install -q ko-pii


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: D:\Project\chatbot\src\.venv\Scripts\python.exe -m pip install --upgrade pip


## 2) 33종 카테고리 정의

ko-pii 가 다루는 33종 라벨. 아래 테스트 끝에서 이 목록 대비 **검출 커버리지**를 계산한다.

In [2]:
# ko-pii 가 지원하는 33종 PII 라벨 (README 기준)
PII_33 = [
    # 결정론적(체크섬/형식 검증)
    "RRN", "FRN", "BUSINESS_REG_NUM", "CORP_REG_NUM",
    "DRIVER_LICENSE", "PASSPORT", "CREDIT_CARD", "PNU",
    # 키워드 앵커(키워드 + 형식)
    "HEALTH_INSURANCE", "PRESCRIPTION_NUM", "DRUG_CODE", "FAX",
    "ACCOUNT_NUM", "EMPLOYEE_ID", "CIVIL_COMPLAINT_NUM", "CASE_NUM",
    # 형식 검증
    "PHONE", "EMAIL", "IP", "URL",
    "POSTAL_CODE", "VEHICLE_NUM", "OFFICIAL_DOCUMENT_NUM",
    # 사전/휴리스틱 + 준식별자
    "PERSON", "ADDRESS", "NATIONALITY", "EDUCATION",
    "MAJOR", "POSITION", "BIRTHDATE", "AGE", "HEIGHT", "WEIGHT",
]
print(f"총 {len(PII_33)}종")
assert len(PII_33) == 33

총 33종


## 3) 장문 샘플 텍스트

가상의 행정 민원 문서를 흉내낸 장문. 33종이 최대한 골고루 들어가도록 구성했다.
**모든 개인정보는 테스트용 가짜 값**이다.

In [3]:
SAMPLE_TEXT = """[민원 접수 및 진료 확인서]

1. 신청인 정보
  - 성명: 홍길동 (국적: 대한민국)
  - 생년월일: 1988-01-01, 만 37세, 신장 175cm, 체중 68kg
  - 주민등록번호: 880101-1234568
  - 외국인등록번호(동반가족): 900101-5678913
  - 주소: 서울특별시 강남구 테헤란로 152, 우편번호 06236
  - 연락처: 010-1234-5678, 팩스: 02-555-1234
  - 이메일: hong.gildong@example.com
  - 최종학력: 서울대학교 졸업 / 전공: 컴퓨터공학 / 직위: 책임연구원

2. 신분/자격 증명
  - 운전면허번호: 11-12-345678-90
  - 여권번호: M12345678
  - 사업자등록번호: 123-45-67890
  - 법인등록번호: 110111-1234567
  - 신용카드번호: 4111-1111-1111-1111
  - 사번: EMP-2024-00123
  - 토지 지번(PNU): 1168010100101230001

3. 진료/보험 정보
  - 건강보험 가입자번호: 1-2345678901
  - 처방전 번호: RX-2024-000123
  - 약품번호(약품코드): 641900010
  - 계좌번호: 110-234-567890 (○○은행)

4. 행정 처리 정보
  - 민원번호: 1AA-2024-1234567
  - 사건번호: 2024가단12345
  - 공문서 문서번호: 강남구-2024-0001234
  - 차량번호: 12가3456

5. 시스템 접근 로그
  - 접속 IP: 192.168.0.42
  - 참고 링크: https://minwon.example.go.kr/case/2024/12345

위 내용은 사실과 다름이 없음을 확인합니다.
"""

print(SAMPLE_TEXT)
print(f"\n(총 길이: {len(SAMPLE_TEXT)} 자)")

[민원 접수 및 진료 확인서]

1. 신청인 정보
  - 성명: 홍길동 (국적: 대한민국)
  - 생년월일: 1988-01-01, 만 37세, 신장 175cm, 체중 68kg
  - 주민등록번호: 880101-1234568
  - 외국인등록번호(동반가족): 900101-5678913
  - 주소: 서울특별시 강남구 테헤란로 152, 우편번호 06236
  - 연락처: 010-1234-5678, 팩스: 02-555-1234
  - 이메일: hong.gildong@example.com
  - 최종학력: 서울대학교 졸업 / 전공: 컴퓨터공학 / 직위: 책임연구원

2. 신분/자격 증명
  - 운전면허번호: 11-12-345678-90
  - 여권번호: M12345678
  - 사업자등록번호: 123-45-67890
  - 법인등록번호: 110111-1234567
  - 신용카드번호: 4111-1111-1111-1111
  - 사번: EMP-2024-00123
  - 토지 지번(PNU): 1168010100101230001

3. 진료/보험 정보
  - 건강보험 가입자번호: 1-2345678901
  - 처방전 번호: RX-2024-000123
  - 약품번호(약품코드): 641900010
  - 계좌번호: 110-234-567890 (○○은행)

4. 행정 처리 정보
  - 민원번호: 1AA-2024-1234567
  - 사건번호: 2024가단12345
  - 공문서 문서번호: 강남구-2024-0001234
  - 차량번호: 12가3456

5. 시스템 접근 로그
  - 접속 IP: 192.168.0.42
  - 참고 링크: https://minwon.example.go.kr/case/2024/12345

위 내용은 사실과 다름이 없음을 확인합니다.


(총 길이: 878 자)


## 4) 검출 실행

`AUDIT` 모드로 실행해 **차단 없이 모든 탐지 결과를 보고**만 받는다.
(STRICT 등은 위험도에 따라 처리 흐름이 바뀔 수 있어, 33종 검출 확인 목적에는 AUDIT 이 적합)

In [4]:
from ko_pii import Anonymizer, ProcessingMode

# AUDIT: 탐지만 보고하고 차단하지 않음 / tokenize: <RRN_1> 형태로 치환
anon = Anonymizer(mode=ProcessingMode.AUDIT, strategy="tokenize")
result = anon.process(SAMPLE_TEXT)

print("===== 비식별화된 텍스트 =====\n")
print(result.text)

===== 비식별화된 텍스트 =====

[민원 접수 및 진료 확인서]

1. 신청인 정보
  - 성명: 홍길동 (국적: 대한민국)
  - 생년월일: 1988-01-01, 만 37세, 신장 175cm, 체중 68kg
  - 주민등록번호: 880101-1234568
  - 외국인등록번호(동반가족): 900101-5678913
  - 주소: 서울특별시 강남구 테헤란로 152, 우편번호 06236
  - 연락처: 010-1234-5678, 팩스: 02-555-1234
  - 이메일: hong.gildong@example.com
  - 최종학력: 서울대학교 졸업 / 전공: 컴퓨터공학 / 직위: 책임연구원

2. 신분/자격 증명
  - 운전면허번호: 11-12-345678-90
  - 여권번호: M12345678
  - 사업자등록번호: 123-45-67890
  - 법인등록번호: 110111-1234567
  - 신용카드번호: 4111-1111-1111-1111
  - 사번: EMP-2024-00123
  - 토지 지번(PNU): 1168010100101230001

3. 진료/보험 정보
  - 건강보험 가입자번호: 1-2345678901
  - 처방전 번호: RX-2024-000123
  - 약품번호(약품코드): 641900010
  - 계좌번호: 110-234-567890 (○○은행)

4. 행정 처리 정보
  - 민원번호: 1AA-2024-1234567
  - 사건번호: 2024가단12345
  - 공문서 문서번호: 강남구-2024-0001234
  - 차량번호: 12가3456

5. 시스템 접근 로그
  - 접속 IP: 192.168.0.42
  - 참고 링크: https://minwon.example.go.kr/case/2024/12345

위 내용은 사실과 다름이 없음을 확인합니다.



## 5) 검출 결과 + 좌표 출력 (핵심)

검출된 항목마다 `라벨`, `좌표[start:end]`, `원문 값`, `신뢰도`, `근거`를 한 줄씩 출력한다.

In [5]:
#------------------------------------------------------------------
# 검출 항목의 근거 문자열 안전 추출
#=> Detection 객체마다 근거 필드 이름이 버전에 따라 다를 수 있어
#    (evidence / legal_basis) 있는 쪽을 골라 문자열로 돌려준다.
#    1) evidence 속성이 있으면 그 값을 사용
#    2) 없으면 legal_basis, 그것도 없으면 빈 문자열
#
# -in: d = ko-pii Detection 객체
#
# -out: 근거 문자열 (없으면 "")
# -out: error = 없음 (항상 문자열 반환)
#------------------------------------------------------------------
def get_evidence(d):
    # 버전별 속성명 차이를 흡수: evidence 우선, 없으면 legal_basis
    return getattr(d, "evidence", None) or getattr(d, "legal_basis", None) or ""


dets = sorted(result.detections, key=lambda d: d.start)  # 좌표 순 정렬
print(f"총 검출 건수: {len(dets)}\n")
print(f"{'LABEL':22} {'[start:end]':>13}  {'원문값':30} conf  근거")
print("-" * 100)
for d in dets:
    coord = f"[{d.start}:{d.end}]"
    print(f"{d.label:22} {coord:>13}  {repr(d.text):30} {d.confidence:.2f}  {get_evidence(d)}")

AttributeError: 'DetectionRecord' object has no attribute 'start'

## 6) 좌표 검증 — 슬라이싱으로 원문 재확인

출력된 좌표가 실제 원문 위치와 일치하는지 `SAMPLE_TEXT[start:end]` 로 잘라 확인한다.

In [ ]:
ok = True
for d in dets:
    sliced = SAMPLE_TEXT[d.start:d.end]  # 좌표로 직접 원문 슬라이싱
    match = (sliced == d.text)
    if not match:
        ok = False
    flag = "OK" if match else "⚠ 불일치"
    print(f"{d.label:22} [{d.start}:{d.end}] 원문슬라이스={repr(sliced):30} {flag}")

print("\n좌표-원문 일치 여부:", "모두 일치 ✅" if ok else "불일치 존재 ⚠")

## 7) 좌표 기반 하이라이트 미리보기

검출 좌표를 이용해 원문에서 해당 구간을 `《라벨:원문》` 형태로 감싸 보여준다.
(뒤에서부터 치환해 앞쪽 좌표가 밀리지 않게 처리)

In [ ]:
#------------------------------------------------------------------
# 좌표로 원문 하이라이트
#=> 검출 구간을 《라벨:원문》 으로 감싸 사람이 눈으로 확인하기 쉽게 만든다.
#    1) start 큰 것부터(뒤에서부터) 치환해야 앞쪽 좌표가 안 밀린다
#    2) 각 구간을 마커로 감싼 새 문자열을 만든다
#
# -in: text  = 원문 문자열
# -in: dets  = Detection 리스트
#
# -out: 하이라이트가 삽입된 문자열
# -out: error = 없음
#------------------------------------------------------------------
def highlight(text, dets):
    out = text
    # 뒤에서부터 치환: 앞 구간을 건드려도 뒤 좌표가 유효하게 유지된다
    for d in sorted(dets, key=lambda x: x.start, reverse=True):
        out = out[:d.start] + f"《{d.label}:{d.text}》" + out[d.end:]
    return out


print(highlight(SAMPLE_TEXT, dets))

## 8) 33종 커버리지 & 요약

In [ ]:
detected_labels = {d.label for d in dets}
found = [x for x in PII_33 if x in detected_labels]
missing = [x for x in PII_33 if x not in detected_labels]
extra = sorted(detected_labels - set(PII_33))  # 33종 목록 밖 라벨(버전차)

print(f"검출된 종류: {len(found)}/33\n")
print("● 검출됨 :", ", ".join(found))
print("\n○ 미검출 :", ", ".join(missing) if missing else "(없음)")
if extra:
    print("\n＋ 목록 외 라벨:", ", ".join(extra))

print("\n===== 라벨별 건수 (summary) =====")
print(result.summary.get("by_label", {}))

In [ ]:
# 결합 위험도(준식별자 조합 위험) 확인 — 버전에 따라 속성 구조가 다를 수 있어 방어적으로 접근
cr = getattr(result, "combined_risk", None)
if cr is not None:
    try:
        print("결합 위험도 :", cr.combined_risk.name)
        print("식별자      :", cr.distinct_identifiers)
        print("준식별자    :", cr.distinct_quasi)
    except Exception as e:
        print("combined_risk 구조 확인 필요:", repr(cr), "/", e)
else:
    print("combined_risk 미제공")

## 9) (참고) 토큰 → 원문 복원

`tokenize` 전략은 Vault 로 원문을 되돌릴 수 있다. 권한자만 접근하는 용도.

In [ ]:
# 검출된 각 토큰을 원문으로 복원해 본다 (Vault.reveal)
import re

tokens = re.findall(r"<[A-Z_]+_\d+>", result.text)
for t in dict.fromkeys(tokens):  # 중복 제거, 순서 유지
    try:
        print(f"{t:16} -> {result.vault.reveal(t)}")
    except Exception as e:
        print(f"{t:16} -> (복원 실패: {e})")

## 10) `detect_all` 함수란? — 순수 "탐지 전용" API

```python
from ko_pii import detect_all as _kopii_detect_all
```

`detect_all` 은 `Anonymizer` 보다 **한 단계 낮은(low-level) 탐지 전용 함수**다.

| 구분 | `Anonymizer.process()` | `detect_all()` |
|------|------------------------|----------------|
| 하는 일 | 탐지 **+ 비식별화(치환) + Vault + 위험도/차단 모드** 전체 파이프라인 | **탐지만** 수행 |
| 반환 | `ProcessingResult` (`.text`, `.detections`, `.vault`, `.summary` ...) | `list[DetectionResult]` (탐지 리스트 그 자체) |
| 용도 | 문서 익명화/토큰화까지 필요할 때 | "어디에 무엇이 있나" **좌표만** 빠르게 뽑을 때 |

**시그니처**

```python
def detect_all(
    text: str,
    include: Optional[Iterable[str]] = None,   # 이 라벨들만 검출
    exclude: Optional[Iterable[str]] = None,   # 이 라벨들은 제외
    *,
    normalize: bool = True,                    # 유니코드 정규화(우회 방지)
) -> list[DetectionResult]
```

- 내부적으로 **모든 detector 를 돌린 뒤**, 겹치는 탐지를 우선순위
  (위험도 → 신뢰도 → 구간 길이 → 시작 위치)로 **충돌 정리(merge)** 한 리스트를 돌려준다.
- 각 `DetectionResult` 는 앞서 본 것과 같은 `label / text / start / end / confidence / legal_basis` 좌표·속성을 갖는다.
- `Anonymizer` 없이 곧바로 **검출 좌표만** 필요할 때 가장 간단하다.

In [ ]:
from ko_pii import detect_all as _kopii_detect_all

# Anonymizer 없이 곧바로 탐지 리스트만 받는다 (치환/Vault 없음)
detections = _kopii_detect_all(SAMPLE_TEXT)
detections = sorted(detections, key=lambda d: d.start)  # 좌표 순 정렬

print(f"detect_all 검출 건수: {len(detections)}\n")
print(f"{'LABEL':22} {'[start:end]':>13}  {'원문값':30} conf")
print("-" * 80)
for d in detections:
    coord = f"[{d.start}:{d.end}]"
    # 좌표로 원문을 다시 잘라 검증까지 겸함
    assert SAMPLE_TEXT[d.start:d.end] == d.text
    print(f"{d.label:22} {coord:>13}  {repr(d.text):30} {d.confidence:.2f}")

### 10-1) `include` / `exclude` 로 특정 라벨만 뽑기

예: 주민번호·연락처·이메일만 관심 있을 때.

In [ ]:
# include: 지정한 라벨만 검출
only = _kopii_detect_all(SAMPLE_TEXT, include={"RRN", "PHONE", "EMAIL"})
print("[include=RRN,PHONE,EMAIL]")
for d in sorted(only, key=lambda d: d.start):
    print(f"  {d.label:8} [{d.start}:{d.end}] {d.text!r}")

# exclude: 준식별자(나이/키/몸무게/생년월일)만 빼고 검출
no_quasi = _kopii_detect_all(SAMPLE_TEXT, exclude={"AGE", "HEIGHT", "WEIGHT", "BIRTHDATE"})
print(f"\n[exclude=AGE,HEIGHT,WEIGHT,BIRTHDATE] -> {len(no_quasi)}건")

### 10-2) `Anonymizer` 결과와 좌표 비교

`detect_all` 과 `Anonymizer(AUDIT).process()` 의 탐지 좌표가 사실상 동일한지 확인.

In [ ]:
# 두 경로의 (라벨, 좌표) 집합 비교
set_detect_all = {(d.label, d.start, d.end) for d in detections}
set_anonymizer = {(d.label, d.start, d.end) for d in result.detections}

print("detect_all 만 :", set_detect_all - set_anonymizer or "(없음)")
print("Anonymizer 만 :", set_anonymizer - set_detect_all or "(없음)")
print("공통 건수     :", len(set_detect_all & set_anonymizer))